# Chess Coach — Agent d'aide à l'apprentissage des ouvertures

**Auteur :** Benoit Girard
**Projet :** agent IA pour l'apprentissage des ouvertures
**Environnement :** Python 3.12, paquet `chess_coach` (uv)

Ce notebook déroule, étape par étape, le raisonnement derrière l'agent. Il
manipule directement les briques du paquet `chess_coach` afin de rendre lisible la
démarche :

0. Configuration et imports
1. Représenter une position d'échecs (FEN)
2. La théorie des ouvertures (livre local)
3. Préparer la base de connaissances Wikichess (chunking)
4. Générer des embeddings
5. L'agent LangGraph : orchestrer les outils
6. Récapitulatif

> Les briques « pures » s'exécutent telles quelles. Les outils nécessitant des
> services externes (Stockfish, Milvus, MongoDB) tournent dans la stack Docker ;
> on en explique ici la logique.

## 0. Configuration et imports

In [ ]:
from pathlib import Path

from chess_coach.config import get_settings

# Le notebook vit dans notebooks/ ; le code Python vit dans backend/.
# On calcule une fois pour toutes la racine du depot, puis on s'y refere.
RACINE = Path.cwd()
if not (RACINE / "docker-compose.yml").exists():
    RACINE = RACINE.parent

DOSSIER_ARTICLES = RACINE / "backend" / "data" / "openings"

settings = get_settings()
print("Racine du depot     :", RACINE)
print("Modele d'embedding  :", settings.embedding_model)
print("Dimension           :", settings.embedding_dim)
print("Collection Milvus   :", settings.milvus_collection)

## 1. Représenter une position d'échecs (FEN)

Avant de raisonner, l'agent doit *comprendre* la position. À l'image de la façon
dont une position est transmise à un LLM (cf. Kaggle Game Arena), on en extrait
une description structurée : trait, coups légaux, échiquier.

In [ ]:
from chess_coach.services.chess_position import STARTING_FEN, describe_position

info = describe_position(STARTING_FEN)
print("Trait aux       :", info.side_to_move)
print(
    "Coups légaux     :",
    len(info.legal_moves_san),
    "->",
    info.legal_moves_san[:6],
    "...",
)
print(info.board_ascii)

**Observation.** Depuis la position initiale, l'agent dispose de 20 coups légaux
et d'un échiquier exploitable. Cette description sera passée aux outils et, le
cas échéant, au LLM de synthèse.

## 2. La théorie des ouvertures (livre local)

Pour un coup *théorique*, on interroge un livre d'ouvertures construit à partir
des grandes lignes. (En production, l'API Lichess enrichit ces coups avec les
statistiques de parties dès qu'un jeton est fourni.)

In [ ]:
from chess_coach.services.opening_book import OpeningBook

result = OpeningBook.lookup(STARTING_FEN)
print("Ouverture :", result.opening_name, f"({result.opening_eco})")
print("Coups théoriques :", [m.san for m in result.moves])

**Observation.** Le livre reconnaît la position et propose les premiers coups
maîtres (e4, d4, c4, Cf3). Les transpositions sont gérées car les positions sont
indexées par leur EPD.

## 3. Préparer la base de connaissances Wikichess (chunking)

La qualité du RAG dépend du découpage. On charge les articles puis on les
segmente en passages avec recouvrement.

In [ ]:
from chess_coach.rag.preprocess import build_chunks, load_articles

articles = load_articles(DOSSIER_ARTICLES)
chunks = build_chunks(articles)
print(f"{len(articles)} articles -> {len(chunks)} chunks")
print("Exemple de chunk :")
print(chunks[0].text[:200], "...")

**Observation.** Les 10 articles produisent plusieurs dizaines de chunks. Chaque
chunk garde le nom de l'ouverture et sa source, ce qui permet de citer l'origine
d'une réponse.

## 4. Générer des embeddings

On transforme un texte en vecteur dense normalisé. Le modèle est multilingue
(les articles sont en français).

In [ ]:
from chess_coach.services.embeddings import EmbeddingService

embedder = EmbeddingService(settings)
vectors = embedder.embed(["La défense sicilienne", "Le gambit dame"])
print("Nombre de vecteurs :", len(vectors))
print("Dimension          :", len(vectors[0]))

**Observation.** Chaque phrase devient un vecteur de 384 dimensions. Indexés
dans Milvus (recherche par produit scalaire ≡ cosinus), ils permettent la
recherche sémantique exposée par `GET /api/v1/vector-search`.

## 5. L'agent LangGraph : orchestrer les outils

L'agent est un graphe de décision. Pour une position :

1. **identify** — décrit la position (FEN valide ?) ;
2. **theory** — cherche les coups théoriques ;
3. **branche** — position connue → *théorie*, sinon → *moteur Stockfish* ;
4. **context** — enrichit avec le RAG Wikichess ;
5. **videos** — propose des vidéos YouTube ;
6. **synthesize** — rédige une recommandation en français ;
7. **persist** — enregistre l'interaction dans MongoDB.

On illustre la dernière étape (la synthèse) à partir d'un état fabriqué, sans
service externe.

In [ ]:
from chess_coach.agent.synthesize import build_template_recommendation

etat = {
    "in_theory": True,
    "opening_name": "Ouverture italienne",
    "opening_eco": "C50",
    "theory_moves": [{"san": "Fc4"}, {"san": "Fb5"}, {"san": "d4"}],
    "passages": [{"text": "L'italienne vise rapidement la case f7."}],
    "videos": [{"title": "Tutoriel"}],
}
print(build_template_recommendation(etat))

**Observation.** À partir des faits collectés par les nœuds, l'agent produit une
recommandation lisible. La couche LLM optionnelle (`LLM_ENABLED`) peut reformuler
cette réponse ; en cas d'échec, le gabarit déterministe reste utilisé.

Pour exécuter l'agent complet de bout en bout, on appelle l'API une fois la stack
démarrée :

```python
import httpx
httpx.post("http://localhost:8000/api/v1/agent", json={"fen": STARTING_FEN}).json()
```

## 6. Récapitulatif

- La position est **comprise** (FEN → description structurée).
- La **théorie** vient du livre local (et de Lichess avec un jeton).
- Hors théorie, **Stockfish** évalue la position.
- Le **RAG Milvus** sur Wikichess apporte le contexte.
- L'**API YouTube** propose des vidéos.
- Le tout est orchestré par **LangGraph**, exposé par **FastAPI**, persisté dans
  **MongoDB** et présenté par une interface **Angular** — l'ensemble étant
  conteneurisé avec **docker-compose**.

La fiche d'auto-évaluation (`docs/auto_evaluation.md`) relie chaque indicateur de
réussite à l'élément qui le satisfait.